# 08. 이벤트/공휴일 효과 분석

티머니 STIS 택시 데이터(D012)를 활용하여 공휴일, 주말, 평일 간 택시 수요 차이를 분석한다.

- 데이터 기간: 2018-01-01 ~ 2026-04-30
- 분석 항목: 공휴일/평일/주말 수요 비교, 명절 전후 패턴, 시간대별 프로파일, 연말연시 패턴, 심야 수요 비교

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 폰트 설정 (Windows 기본)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac 사용 시 아래 주석 해제
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

## 1. 한국 공휴일 리스트 하드코딩 (2018~2026)

In [ ]:
# 한국 공휴일 리스트 (2018~2026)
# 설날, 추석은 음력 기준이므로 매년 날짜가 다름
# 대체공휴일 포함

holidays_raw = {
    # --- 2018 ---
    '2018-01-01': '신정',
    '2018-02-15': '설날 전날', '2018-02-16': '설날', '2018-02-17': '설날 다음날',
    '2018-03-01': '삼일절',
    '2018-05-05': '어린이날', '2018-05-07': '대체공휴일(어린이날)',
    '2018-05-22': '석가탄신일',
    '2018-06-06': '현충일',
    '2018-06-13': '지방선거',
    '2018-08-15': '광복절',
    '2018-09-23': '추석 전날', '2018-09-24': '추석', '2018-09-25': '추석 다음날',
    '2018-09-26': '대체공휴일(추석)',
    '2018-10-03': '개천절',
    '2018-10-09': '한글날',
    '2018-12-25': '크리스마스',

    # --- 2019 ---
    '2019-01-01': '신정',
    '2019-02-04': '설날 전날', '2019-02-05': '설날', '2019-02-06': '설날 다음날',
    '2019-03-01': '삼일절',
    '2019-05-05': '어린이날', '2019-05-06': '대체공휴일(어린이날)',
    '2019-05-12': '석가탄신일',
    '2019-06-06': '현충일',
    '2019-08-15': '광복절',
    '2019-09-12': '추석 전날', '2019-09-13': '추석', '2019-09-14': '추석 다음날',
    '2019-10-03': '개천절',
    '2019-10-09': '한글날',
    '2019-12-25': '크리스마스',

    # --- 2020 ---
    '2020-01-01': '신정',
    '2020-01-24': '설날 전날', '2020-01-25': '설날', '2020-01-26': '설날 다음날',
    '2020-01-27': '대체공휴일(설날)',
    '2020-03-01': '삼일절',
    '2020-04-15': '총선',
    '2020-04-30': '석가탄신일',
    '2020-05-05': '어린이날',
    '2020-06-06': '현충일',
    '2020-08-15': '광복절',
    '2020-08-17': '대체공휴일(광복절)',
    '2020-09-30': '추석 전날', '2020-10-01': '추석', '2020-10-02': '추석 다음날',
    '2020-10-03': '개천절',
    '2020-10-09': '한글날',
    '2020-12-25': '크리스마스',

    # --- 2021 ---
    '2021-01-01': '신정',
    '2021-02-11': '설날 전날', '2021-02-12': '설날', '2021-02-13': '설날 다음날',
    '2021-03-01': '삼일절',
    '2021-05-05': '어린이날',
    '2021-05-19': '석가탄신일',
    '2021-06-06': '현충일',
    '2021-08-15': '광복절',
    '2021-08-16': '대체공휴일(광복절)',
    '2021-09-20': '추석 전날', '2021-09-21': '추석', '2021-09-22': '추석 다음날',
    '2021-10-03': '개천절',
    '2021-10-04': '대체공휴일(개천절)',
    '2021-10-09': '한글날',
    '2021-10-11': '대체공휴일(한글날)',
    '2021-12-25': '크리스마스',

    # --- 2022 ---
    '2022-01-01': '신정',
    '2022-01-31': '설날 전날', '2022-02-01': '설날', '2022-02-02': '설날 다음날',
    '2022-03-01': '삼일절',
    '2022-03-09': '대선',
    '2022-05-05': '어린이날',
    '2022-05-08': '석가탄신일',
    '2022-06-01': '지방선거',
    '2022-06-06': '현충일',
    '2022-08-15': '광복절',
    '2022-09-09': '추석 전날', '2022-09-10': '추석', '2022-09-11': '추석 다음날',
    '2022-09-12': '대체공휴일(추석)',
    '2022-10-03': '개천절',
    '2022-10-09': '한글날',
    '2022-10-10': '대체공휴일(한글날)',
    '2022-12-25': '크리스마스',

    # --- 2023 ---
    '2023-01-01': '신정',
    '2023-01-21': '설날 전날', '2023-01-22': '설날', '2023-01-23': '설날 다음날',
    '2023-01-24': '대체공휴일(설날)',
    '2023-03-01': '삼일절',
    '2023-05-05': '어린이날',
    '2023-05-27': '석가탄신일',
    '2023-05-29': '대체공휴일(석가탄신일)',
    '2023-06-06': '현충일',
    '2023-08-15': '광복절',
    '2023-09-28': '추석 전날', '2023-09-29': '추석', '2023-09-30': '추석 다음날',
    '2023-10-03': '개천절',
    '2023-10-09': '한글날',
    '2023-12-25': '크리스마스',

    # --- 2024 ---
    '2024-01-01': '신정',
    '2024-02-09': '설날 전날', '2024-02-10': '설날', '2024-02-11': '설날 다음날',
    '2024-02-12': '대체공휴일(설날)',
    '2024-03-01': '삼일절',
    '2024-04-10': '총선',
    '2024-05-05': '어린이날', '2024-05-06': '대체공휴일(어린이날)',
    '2024-05-15': '석가탄신일',
    '2024-06-06': '현충일',
    '2024-08-15': '광복절',
    '2024-09-16': '추석 전날', '2024-09-17': '추석', '2024-09-18': '추석 다음날',
    '2024-10-03': '개천절',
    '2024-10-09': '한글날',
    '2024-12-25': '크리스마스',

    # --- 2025 ---
    '2025-01-01': '신정',
    '2025-01-28': '설날 전날', '2025-01-29': '설날', '2025-01-30': '설날 다음날',
    '2025-03-01': '삼일절',
    '2025-03-03': '대체공휴일(삼일절)',
    '2025-05-05': '어린이날', '2025-05-06': '대체공휴일(석가탄신일)',
    '2025-06-06': '현충일',
    '2025-08-15': '광복절',
    '2025-10-03': '개천절',
    '2025-10-05': '추석 전날', '2025-10-06': '추석', '2025-10-07': '추석 다음날',
    '2025-10-08': '대체공휴일(추석)',
    '2025-10-09': '한글날',
    '2025-12-25': '크리스마스',

    # --- 2026 ---
    '2026-01-01': '신정',
    '2026-02-16': '설날 전날', '2026-02-17': '설날', '2026-02-18': '설날 다음날',
    '2026-03-01': '삼일절',
    '2026-03-02': '대체공휴일(삼일절)',
    '2026-04-08': '재보궐선거',
    '2026-05-05': '어린이날',
    '2026-05-24': '석가탄신일',
    '2026-05-25': '대체공휴일(석가탄신일)',
    '2026-06-06': '현충일',
    '2026-08-15': '광복절',
    '2026-08-17': '대체공휴일(광복절)',
    '2026-09-24': '추석 전날', '2026-09-25': '추석', '2026-09-26': '추석 다음날',
    '2026-10-03': '개천절',
    '2026-10-05': '대체공휴일(개천절)',
    '2026-10-09': '한글날',
    '2026-12-25': '크리스마스'
}

# 날짜 변환
holidays = {pd.Timestamp(k): v for k, v in holidays_raw.items()}
holiday_dates = set(holidays.keys())

print(f'등록된 공휴일 수: {len(holidays)}개')
print(f'기간: {min(holidays.keys()).strftime("%Y-%m-%d")} ~ {max(holidays.keys()).strftime("%Y-%m-%d")}')

In [ ]:
# 설날/추석 당일 날짜 (명절 전후 패턴 분석용)
seollal_dates = [
    pd.Timestamp('2018-02-16'), pd.Timestamp('2019-02-05'), pd.Timestamp('2020-01-25'),
    pd.Timestamp('2021-02-12'), pd.Timestamp('2022-02-01'), pd.Timestamp('2023-01-22'),
    pd.Timestamp('2024-02-10'), pd.Timestamp('2025-01-29'), pd.Timestamp('2026-02-17')
]

chuseok_dates = [
    pd.Timestamp('2018-09-24'), pd.Timestamp('2019-09-13'), pd.Timestamp('2020-10-01'),
    pd.Timestamp('2021-09-21'), pd.Timestamp('2022-09-10'), pd.Timestamp('2023-09-29'),
    pd.Timestamp('2024-09-17'), pd.Timestamp('2025-10-06'), pd.Timestamp('2026-09-25')
]

print(f'설날 {len(seollal_dates)}개년, 추석 {len(chuseok_dates)}개년 등록')

## 2. 데이터 로드 및 일별 승차건수 집계 + 라벨링

In [ ]:
# 데이터 로드
df = pd.read_csv(r"C:\Users\admin\Desktop\작업 폴더\tmoney\DC_TBYXD012.csv")
print(f'전체 레코드 수: {len(df):,}')

# RIDE_DTIME 파싱
df['RIDE_DTIME'] = pd.to_datetime(df['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df = df.dropna(subset=['RIDE_DTIME'])

# 파생 컬럼
df['date'] = df['RIDE_DTIME'].dt.date
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['RIDE_DTIME'].dt.hour
df['dayofweek'] = df['RIDE_DTIME'].dt.dayofweek  # 0=월 ~ 6=일

print(f'유효 레코드 수: {len(df):,}')
print(f'기간: {df["date"].min().strftime("%Y-%m-%d")} ~ {df["date"].max().strftime("%Y-%m-%d")}')

In [ ]:
# 일별 승차건수 집계
daily = df.groupby('date').size().reset_index(name='ride_count')
daily['dayofweek'] = daily['date'].dt.dayofweek

# 공휴일/평일/주말 라벨링
def classify_day(row):
    if row['date'] in holiday_dates:
        return '공휴일'
    elif row['dayofweek'] >= 5:  # 토(5), 일(6)
        return '주말'
    else:
        return '평일'

daily['day_type'] = daily.apply(classify_day, axis=1)

print('일별 데이터 라벨링 완료')
print(daily['day_type'].value_counts())

## 3. 공휴일 vs 평일 vs 주말 수요 비교

In [ ]:
# 유형별 평균 승차건수
day_type_stats = daily.groupby('day_type')['ride_count'].agg(['mean', 'median', 'std', 'count']).round(0)
day_type_stats = day_type_stats.reindex(['평일', '주말', '공휴일'])
print(day_type_stats)

# 바차트
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4472C4', '#ED7D31', '#A5A5A5']
order = ['평일', '주말', '공휴일']
means = [day_type_stats.loc[t, 'mean'] for t in order]

bars = ax.bar(order, means, color=colors, edgecolor='black', linewidth=0.5, width=0.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(means) * 0.01,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=11)

ax.set_ylabel('평균 일일 승차건수')
ax.set_title('공휴일 vs 평일 vs 주말 평균 택시 수요')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 4. 주요 명절별 수요 패턴: 설날/추석 전후 (D-3 ~ D+3)

In [ ]:
def get_holiday_window(center_dates, daily_df, window=3, label='명절'):
    """명절 당일 기준 D-window ~ D+window 수요 집계 (연도별 평균)"""
    records = []
    for center in center_dates:
        year = center.year
        for offset in range(-window, window + 1):
            target_date = center + timedelta(days=offset)
            row = daily_df[daily_df['date'] == target_date]
            if not row.empty:
                records.append({
                    'year': year,
                    'offset': offset,
                    'ride_count': row['ride_count'].values[0]
                })
    result = pd.DataFrame(records)
    if result.empty:
        return result
    return result.groupby('offset')['ride_count'].mean().reset_index()

seollal_pattern = get_holiday_window(seollal_dates, daily, window=3)
chuseok_pattern = get_holiday_window(chuseok_dates, daily, window=3)

# 라인차트
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title in zip(axes, [seollal_pattern, chuseok_pattern], ['설날', '추석']):
    if data.empty:
        ax.set_title(f'{title} - 데이터 없음')
        continue
    ax.plot(data['offset'], data['ride_count'], marker='o', linewidth=2, color='#4472C4')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label=f'{title} 당일')
    ax.set_xlabel('D-day 기준 오프셋 (일)')
    ax.set_ylabel('평균 승차건수')
    ax.set_title(f'{title} 전후 택시 수요 변화 (D-3 ~ D+3)')
    ax.set_xticks(range(-3, 4))
    ax.set_xticklabels([f'D{i:+d}' if i != 0 else 'D-day' for i in range(-3, 4)])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 시간대별 수요 프로파일: 공휴일 vs 평일 vs 주말

In [ ]:
# 각 레코드에 day_type 부여
df['day_type'] = df['date'].map(lambda d: '공휴일' if d in holiday_dates else ('주말' if d.dayofweek >= 5 else '평일'))

# 시간대 x day_type별 평균 승차건수 (일평균)
# 먼저 일별-시간대별 건수 집계
hourly_daily = df.groupby(['date', 'day_type', 'hour']).size().reset_index(name='count')

# day_type별 시간대 평균
hourly_profile = hourly_daily.groupby(['day_type', 'hour'])['count'].mean().reset_index()

# 시각화
fig, ax = plt.subplots(figsize=(12, 5))
colors_map = {'평일': '#4472C4', '주말': '#ED7D31', '공휴일': '#A5A5A5'}

for dtype in ['평일', '주말', '공휴일']:
    subset = hourly_profile[hourly_profile['day_type'] == dtype]
    ax.plot(subset['hour'], subset['count'], marker='o', markersize=4,
            linewidth=2, label=dtype, color=colors_map[dtype])

ax.set_xlabel('시간대')
ax.set_ylabel('평균 승차건수')
ax.set_title('시간대별 택시 수요 프로파일 (공휴일 vs 평일 vs 주말)')
ax.set_xticks(range(24))
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 6. 연말연시 (12.24 ~ 1.1) 특별 패턴

In [ ]:
# 연말연시 기간 추출 (12/24 ~ 1/1)
def is_year_end(date):
    """12월 24일 ~ 1월 1일 여부"""
    m, d = date.month, date.day
    return (m == 12 and d >= 24) or (m == 1 and d <= 1)

daily['is_year_end'] = daily['date'].apply(is_year_end)
year_end = daily[daily['is_year_end']].copy()

# 연말연시 시즌 라벨: 12/24 기준으로 시즌 연도 부여
year_end['season'] = year_end['date'].apply(lambda d: d.year if d.month == 12 else d.year - 1)
year_end['day_label'] = year_end['date'].apply(
    lambda d: f"{d.month}/{d.day}"
)

# 날짜 순서를 위한 정렬 키 (12/24=0, 12/25=1, ... 1/1=8)
def year_end_order(d):
    if d.month == 12:
        return d.day - 24
    else:
        return 8  # 1/1

year_end['order'] = year_end['date'].apply(year_end_order)

# 시즌별 평균
ye_avg = year_end.groupby('order').agg(
    ride_count=('ride_count', 'mean'),
    day_label=('day_label', 'first')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ye_avg['order'], ye_avg['ride_count'], marker='o', linewidth=2, color='#C00000')
ax.set_xticks(ye_avg['order'])
ax.set_xticklabels(ye_avg['day_label'])
ax.set_xlabel('날짜')
ax.set_ylabel('평균 승차건수')
ax.set_title('연말연시(12/24~1/1) 택시 수요 패턴')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.3)

# 크리스마스, 신정 표시
for idx, row in ye_avg.iterrows():
    if row['day_label'] in ['12/25', '1/1']:
        ax.annotate(row['day_label'], (row['order'], row['ride_count']),
                    textcoords='offset points', xytext=(0, 10), ha='center',
                    fontsize=9, color='red')

plt.tight_layout()
plt.show()

## 7. 금요일 vs 토요일 vs 공휴일 전날 심야 수요 비교

In [ ]:
# 심야 시간대 정의: 22시 ~ 03시 (다음날)
late_night_hours = [22, 23, 0, 1, 2, 3]

# 공휴일 전날 날짜 계산
eve_of_holiday = set()
for hd in holiday_dates:
    eve = hd - timedelta(days=1)
    # 전날이 이미 공휴일이거나 주말이 아닌 경우만
    if eve not in holiday_dates and eve.dayofweek < 5:
        eve_of_holiday.add(eve)

# 각 레코드에 심야 카테고리 부여
df_late = df[df['hour'].isin(late_night_hours)].copy()

def classify_late_night(row):
    """심야 수요 분류: 금요일 / 토요일 / 공휴일 전날"""
    d = row['date']
    dow = d.dayofweek
    if d in eve_of_holiday:
        return '공휴일 전날'
    elif dow == 4:  # 금요일
        return '금요일'
    elif dow == 5:  # 토요일
        return '토요일'
    else:
        return None

df_late['late_type'] = df_late.apply(classify_late_night, axis=1)
df_late = df_late.dropna(subset=['late_type'])

# 시간대별 평균 (일평균 기준)
late_daily = df_late.groupby(['date', 'late_type', 'hour']).size().reset_index(name='count')
late_profile = late_daily.groupby(['late_type', 'hour'])['count'].mean().reset_index()

# 시간 순서 재정렬 (22, 23, 0, 1, 2, 3)
hour_order = {22: 0, 23: 1, 0: 2, 1: 3, 2: 4, 3: 5}
late_profile['hour_order'] = late_profile['hour'].map(hour_order)
late_profile = late_profile.sort_values(['late_type', 'hour_order'])

# 바차트: 유형별 심야 총 평균 건수
late_total = df_late.groupby(['date', 'late_type']).size().reset_index(name='count')
late_avg = late_total.groupby('late_type')['count'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (좌) 유형별 심야 평균 총건수 바차트
ax = axes[0]
order_labels = ['금요일', '토요일', '공휴일 전날']
vals = [late_avg.get(l, 0) for l in order_labels]
bar_colors = ['#4472C4', '#ED7D31', '#A5A5A5']
bars = ax.bar(order_labels, vals, color=bar_colors, edgecolor='black', linewidth=0.5, width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.01,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('평균 심야 승차건수 (22~03시)')
ax.set_title('심야 시간대 수요 비교')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# (우) 시간대별 라인차트
ax = axes[1]
line_colors = {'금요일': '#4472C4', '토요일': '#ED7D31', '공휴일 전날': '#A5A5A5'}
hour_labels = ['22시', '23시', '0시', '1시', '2시', '3시']
for ltype in order_labels:
    subset = late_profile[late_profile['late_type'] == ltype].sort_values('hour_order')
    if not subset.empty:
        ax.plot(subset['hour_order'], subset['count'], marker='o', linewidth=2,
                label=ltype, color=line_colors[ltype])

ax.set_xticks(range(6))
ax.set_xticklabels(hour_labels)
ax.set_xlabel('시간대')
ax.set_ylabel('평균 승차건수')
ax.set_title('심야 시간대별 수요 프로파일')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

## 8. 요약

In [ ]:
# 주요 통계 요약
print('=' * 60)
print('이벤트/공휴일 효과 분석 요약')
print('=' * 60)

# 유형별 평균
print('\n[1] 유형별 일평균 승차건수')
for dtype in ['평일', '주말', '공휴일']:
    stats = daily[daily['day_type'] == dtype]['ride_count']
    print(f'  {dtype}: 평균 {stats.mean():,.0f}건 / 중앙값 {stats.median():,.0f}건 / 일수 {len(stats):,}일')

# 평일 대비 비율
weekday_avg = daily[daily['day_type'] == '평일']['ride_count'].mean()
weekend_avg = daily[daily['day_type'] == '주말']['ride_count'].mean()
holiday_avg = daily[daily['day_type'] == '공휴일']['ride_count'].mean()

print(f'\n[2] 평일 대비 수요 비율')
print(f'  주말/평일: {weekend_avg / weekday_avg:.1%}')
print(f'  공휴일/평일: {holiday_avg / weekday_avg:.1%}')

# 설날/추석 당일 평균
print(f'\n[3] 명절 당일 평균 수요')
for name, dates in [('설날', seollal_dates), ('추석', chuseok_dates)]:
    vals = daily[daily['date'].isin(dates)]['ride_count']
    if not vals.empty:
        print(f'  {name} 당일: 평균 {vals.mean():,.0f}건 (평일 대비 {vals.mean() / weekday_avg:.1%})')

# 심야 수요
print(f'\n[4] 심야(22~03시) 평균 수요')
for ltype in ['금요일', '토요일', '공휴일 전날']:
    val = late_avg.get(ltype, 0)
    print(f'  {ltype}: {val:,.0f}건')

print('\n' + '=' * 60)